In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [3]:
train_dir = r'..\data\Training'
test_dir = r'..\data\Testing'

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

print("Training directory:", train_dir)
print("Testing directory:", test_dir)

Training directory: ..\data\Training
Testing directory: ..\data\Testing


In [4]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 5600 files belonging to 4 classes.
Using 1120 files for validation.


In [5]:
class_names = train_dataset.class_names

print("Class names:", class_names)
print("Number of classes:", len(class_names))

Class names: ['glioma', 'meningioma', 'notumor', 'pituitary']
Number of classes: 4


In [6]:
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Test class names:", test_dataset.class_names)

Found 1600 files belonging to 4 classes.
Test class names: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [7]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_dataset.cache().shuffle(1000).prefetch(
    buffer_size=AUTOTUNE
)

val_ds = val_dataset.cache().prefetch(
    buffer_size=AUTOTUNE
)

test_ds = test_dataset.cache().prefetch(
    buffer_size=AUTOTUNE
)

print("Datasets are ready.")

Datasets are ready.


In [8]:
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("First 10 labels:", labels.numpy()[:10])

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
First 10 labels: [2 0 3 0 2 1 3 2 2 1]


In [9]:
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

print("VGG16 base model loaded successfully.")

VGG16 base model loaded successfully.


In [11]:
# Freeze the pretrained VGG16 layers
base_model.trainable = False

print("VGG16 layers frozen.")
print("Total layers:", len(base_model.layers))

VGG16 layers frozen.
Total layers: 19


In [12]:
from tensorflow.keras import layers, models

model3 = models.Sequential([
    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(4, activation="softmax")
])

model3.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,847,044 (56.64 MB)

 Trainable params: 132,356 (517.02 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [14]:
model3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model 3 compiled successfully.")

Model 3 compiled successfully.


In [ ]:
history3 = model3.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 2857s 20s/step - accuracy: 0.4661 - loss: 3.0782 - val_accuracy: 0.7821 - val_loss: 0.6303
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 2908s 20s/step - accuracy: 0.6777 - loss: 1.1748 - val_accuracy: 0.8357 - val_loss: 0.4526
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 3922s 28s/step - accuracy: 0.7511 - loss: 0.7871 - val_accuracy: 0.8696 - val_loss: 0.3730
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 3111s 22s/step - accuracy: 0.7761 - loss: 0.6237 - val_accuracy: 0.8911 - val_loss: 0.3333
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 2931s 21s/step - accuracy: 0.8078 - loss: 0.5440 - val_accuracy: 0.8911 - val_loss: 0.3166
Epoch 6/10
 74/140 ━━━━━━━━━━━━━━━━━━━━ 14:42 13s/step - accuracy: 0.8073 - loss: 0.5141